In [1]:
import os, gc, importlib.util
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
VOL_TEST = "/tmp/vol_test"

def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

def load_module(name, fname):
    path = find_file(fname)
    print(f"{fname:28s} -> {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

pp = load_module("pp", "03_preprocess_mri.py")     # preprocessing (notebook 02)
tb = load_module("tb", "06_train_backbone.py")     # definisi model (notebook 03c)
LABELS = tb.LABELS

# (backbone, file bobot) — ensemble dipilih berdasarkan data validasi
MODELS = [
    ("efficientnet_b0", "efficientnet_b0_best.pt"),           # seed 42 (LB 0.842)
    ("efficientnet_b0", "effb0_s2_best.pt"),                  # seed 7
    ("vit_small_patch14_dinov2.lvd142m", "dinov2s_v2_best.pt"),
]
for bb, w in MODELS:
    print(f"{w:28s} -> {find_file(w)}")

03_preprocess_mri.py         -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/03_preprocess_mri.py
06_train_backbone.py         -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03c-train-dinov2-v2/06_train_backbone.py
efficientnet_b0_best.pt      -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03-train-baseline/efficientnet_b0_best.pt
effb0_s2_best.pt             -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03c-train-dinov2-v2/effb0_s2_best.pt
dinov2s_v2_best.pt           -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03c-train-dinov2-v2/dinov2s_v2_best.pt


In [2]:
pp.SPLIT, pp.OUT_DIR = "test", VOL_TEST
os.makedirs(VOL_TEST, exist_ok=True)

series = pd.read_csv(f"{COMP}/test_series.csv")
failed = 0
for uid, g in tqdm(series.groupby("StudyInstanceUID"), desc="Preprocess test"):
    try:
        pp.process_study(uid, g)
    except Exception as e:
        failed += 1
        print("Gagal:", uid, type(e).__name__, e)

test = pd.read_csv(f"{COMP}/test.csv")
uids = test["StudyInstanceUID"].tolist()
ok = [u for u in uids if os.path.exists(f"{VOL_TEST}/{u}.npz")]
print(f"Test: {len(uids)} studi | volume tersedia: {len(ok)} | gagal: {failed}")
assert len(ok) >= 0.95 * len(uids), "Terlalu banyak studi gagal diproses, jangan disubmit"

Preprocess test:   0%|          | 0/3 [00:00<?, ?it/s]

Test: 3 studi | volume tersedia: 3 | gagal: 0


In [3]:
ds = tb.KneeDS(ok, np.full((len(ok), len(LABELS)), np.nan, np.float32), VOL_TEST, train=False)
dl = torch.utils.data.DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)

preds = []
for backbone, wfile in MODELS:
    net = tb.KneeNet(backbone, pretrained=False)                
    net.load_state_dict(torch.load(find_file(wfile), map_location="cpu"))
    net = net.cuda().eval()
    if "vit" not in backbone:
        net = net.to(memory_format=torch.channels_last)
    p = tb.predict(net, dl)
    preds.append(p)
    print(f"{wfile:28s} selesai | rata-rata prediksi {p.mean():.3f}")
    del net; gc.collect(); torch.cuda.empty_cache()

p_ens = np.mean(preds, axis=0)
print("Bentuk prediksi ensemble:", p_ens.shape)

efficientnet_b0_best.pt      selesai | rata-rata prediksi 0.182
effb0_s2_best.pt             selesai | rata-rata prediksi 0.139
dinov2s_v2_best.pt           selesai | rata-rata prediksi 0.232
Bentuk prediksi ensemble: (3, 12)


In [4]:
pred = pd.DataFrame(p_ens, columns=LABELS)
pred.insert(0, "StudyInstanceUID", ok)

sub = test[["StudyInstanceUID"]].merge(pred, on="StudyInstanceUID", how="left")
n_missing = sub[LABELS].isna().any(axis=1).sum()
sub[LABELS] = sub[LABELS].fillna(0.5)

sample = pd.read_csv(f"{COMP}/sample_submission.csv")
sub = sub[sample.columns]
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Studi tanpa prediksi (diisi 0.5):", n_missing)
print("Shape submission:", sub.shape, "| shape sample:", sample.shape)
print(sub.head())

Studi tanpa prediksi (diisi 0.5): 0
Shape submission: (3, 13) | shape sample: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.008127  0.028738   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.517497  0.074145   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.004158  0.013077   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.237267          0.066681   0.344553    0.146668  0.310213   
1         0.408427          0.375105   0.054437    0.085961  0.141985   
2         0.507192          0.119478   0.298373    0.092140  0.519157   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.565795   0.031327  0.540384   0.006701  0.002497  
1  0.184786   0.511216  0.048603   0.152769  0.066100  
2  0.074136   0.017350  0.049631   0.011159  0.018272  
